# Data Reading

In [1]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [2]:
df = spark.read.format("delta")\
    .load("s3://learn-databricks-project-e2e-1-bronze/orders")
display(df)

,order_id,customer_id,product_id,order_date,quantity,total_amount,_rescued_data
0,O00001,C00710,P0159,2023-03-22,3,2022.87,None
1,O00002,C00954,P0036,2023-06-30,2,3560.74,None
2,O00003,C01578,P0427,2023-11-06,3,5903.52,None
3,O00004,C00962,P0332,2024-02-27,3,4107.99,None
4,O00005,C00156,P0038,2024-10-13,5,5784.95,None
5,O00006,C00521,P0174,2023-05-17,5,407.75,None
6,O00007,C00982,P0352,2024-01-18,4,4907.64,None
7,O00008,C00976,P0172,2023-01-10,4,7037.88,None
8,O00009,C01001,P0238,2023-04-20,3,4076.97,None
9,O00010,C00702,P0258,2023-07-07,4,5695.64,None


In [3]:
df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- _rescued_data: string (nullable = true)



In [4]:
df = df.withColumnRenamed("_rescued_data", "rescued_data")

In [5]:
df = df.drop("rescued_data")
display(df)

,order_id,customer_id,product_id,order_date,quantity,total_amount
0,O00001,C00710,P0159,2023-03-22,3,2022.87
1,O00002,C00954,P0036,2023-06-30,2,3560.74
2,O00003,C01578,P0427,2023-11-06,3,5903.52
3,O00004,C00962,P0332,2024-02-27,3,4107.99
4,O00005,C00156,P0038,2024-10-13,5,5784.95
5,O00006,C00521,P0174,2023-05-17,5,407.75
6,O00007,C00982,P0352,2024-01-18,4,4907.64
7,O00008,C00976,P0172,2023-01-10,4,7037.88
8,O00009,C01001,P0238,2023-04-20,3,4076.97
9,O00010,C00702,P0258,2023-07-07,4,5695.64


In [6]:
df = df.withColumn("order_date", to_timestamp(col("order_date")))
df.show()

+--------+-----------+----------+-------------------+--------+------------+
|order_id|customer_id|product_id|         order_date|quantity|total_amount|
+--------+-----------+----------+-------------------+--------+------------+
|  O00001|     C00710|     P0159|2023-03-22 00:00:00|       3|     2022.87|
|  O00002|     C00954|     P0036|2023-06-30 00:00:00|       2|     3560.74|
|  O00003|     C01578|     P0427|2023-11-06 00:00:00|       3|     5903.52|
|  O00004|     C00962|     P0332|2024-02-27 00:00:00|       3|     4107.99|
|  O00005|     C00156|     P0038|2024-10-13 00:00:00|       5|     5784.95|
|  O00006|     C00521|     P0174|2023-05-17 00:00:00|       5|      407.75|
|  O00007|     C00982|     P0352|2024-01-18 00:00:00|       4|     4907.64|
|  O00008|     C00976|     P0172|2023-01-10 00:00:00|       4|     7037.88|
|  O00009|     C01001|     P0238|2023-04-20 00:00:00|       3|     4076.97|
|  O00010|     C00702|     P0258|2023-07-07 00:00:00|       4|     5695.64|
|  O00011|  

In [7]:
df = df.withColumn("year", year(col("order_date")))\
    .withColumn("month", month(col("order_date")))\
    .withColumn("day", dayofmonth(col("order_date")))
    
df.show()

+--------+-----------+----------+-------------------+--------+------------+----+-----+---+
|order_id|customer_id|product_id|         order_date|quantity|total_amount|year|month|day|
+--------+-----------+----------+-------------------+--------+------------+----+-----+---+
|  O00001|     C00710|     P0159|2023-03-22 00:00:00|       3|     2022.87|2023|    3| 22|
|  O00002|     C00954|     P0036|2023-06-30 00:00:00|       2|     3560.74|2023|    6| 30|
|  O00003|     C01578|     P0427|2023-11-06 00:00:00|       3|     5903.52|2023|   11|  6|
|  O00004|     C00962|     P0332|2024-02-27 00:00:00|       3|     4107.99|2024|    2| 27|
|  O00005|     C00156|     P0038|2024-10-13 00:00:00|       5|     5784.95|2024|   10| 13|
|  O00006|     C00521|     P0174|2023-05-17 00:00:00|       5|      407.75|2023|    5| 17|
|  O00007|     C00982|     P0352|2024-01-18 00:00:00|       4|     4907.64|2024|    1| 18|
|  O00008|     C00976|     P0172|2023-01-10 00:00:00|       4|     7037.88|2023|    1| 10|

In [8]:
df1 = df.withColumn("flag",dense_rank().over(Window.partitionBy("year").orderBy(desc("total_amount"))))
df1.show()

+--------+-----------+----------+-------------------+--------+------------+----+-----+---+----+
|order_id|customer_id|product_id|         order_date|quantity|total_amount|year|month|day|flag|
+--------+-----------+----------+-------------------+--------+------------+----+-----+---+----+
|  O00957|     C01449|     P0498|2023-10-05 00:00:00|       5|      9952.9|2023|   10|  5|   1|
|  O01765|     C01515|     P0498|2023-05-11 00:00:00|       5|      9952.9|2023|    5| 11|   1|
|  O03502|     C00805|     P0498|2023-09-24 00:00:00|       5|      9952.9|2023|    9| 24|   1|
|  O03660|     C01001|     P0498|2023-10-08 00:00:00|       5|      9952.9|2023|   10|  8|   1|
|  O06790|     C01819|     P0498|2023-02-27 00:00:00|       5|      9952.9|2023|    2| 27|   1|
|  O03989|     C00631|     P0440|2023-11-20 00:00:00|       5|     9916.25|2023|   11| 20|   2|
|  O07763|     C00471|     P0440|2023-04-12 00:00:00|       5|     9916.25|2023|    4| 12|   2|
|  O03272|     C01879|     P0165|2023-02

In [9]:
df2 = df1.withColumn("rank_flag", rank().over(Window.partitionBy("year").orderBy(desc("total_amount"))))
df2.show()

+--------+-----------+----------+-------------------+--------+------------+----+-----+---+----+---------+
|order_id|customer_id|product_id|         order_date|quantity|total_amount|year|month|day|flag|rank_flag|
+--------+-----------+----------+-------------------+--------+------------+----+-----+---+----+---------+
|  O00957|     C01449|     P0498|2023-10-05 00:00:00|       5|      9952.9|2023|   10|  5|   1|        1|
|  O01765|     C01515|     P0498|2023-05-11 00:00:00|       5|      9952.9|2023|    5| 11|   1|        1|
|  O03502|     C00805|     P0498|2023-09-24 00:00:00|       5|      9952.9|2023|    9| 24|   1|        1|
|  O03660|     C01001|     P0498|2023-10-08 00:00:00|       5|      9952.9|2023|   10|  8|   1|        1|
|  O06790|     C01819|     P0498|2023-02-27 00:00:00|       5|      9952.9|2023|    2| 27|   1|        1|
|  O03989|     C00631|     P0440|2023-11-20 00:00:00|       5|     9916.25|2023|   11| 20|   2|        6|
|  O07763|     C00471|     P0440|2023-04-12 00

In [10]:
df3 = df2.withColumn("row_flag", row_number().over(Window.partitionBy("year").orderBy(desc("total_amount"))))
df3.show()

+--------+-----------+----------+-------------------+--------+------------+----+-----+---+----+---------+--------+
|order_id|customer_id|product_id|         order_date|quantity|total_amount|year|month|day|flag|rank_flag|row_flag|
+--------+-----------+----------+-------------------+--------+------------+----+-----+---+----+---------+--------+
|  O00957|     C01449|     P0498|2023-10-05 00:00:00|       5|      9952.9|2023|   10|  5|   1|        1|       1|
|  O01765|     C01515|     P0498|2023-05-11 00:00:00|       5|      9952.9|2023|    5| 11|   1|        1|       2|
|  O03502|     C00805|     P0498|2023-09-24 00:00:00|       5|      9952.9|2023|    9| 24|   1|        1|       3|
|  O03660|     C01001|     P0498|2023-10-08 00:00:00|       5|      9952.9|2023|   10|  8|   1|        1|       4|
|  O06790|     C01819|     P0498|2023-02-27 00:00:00|       5|      9952.9|2023|    2| 27|   1|        1|       5|
|  O03989|     C00631|     P0440|2023-11-20 00:00:00|       5|     9916.25|2023|

# Classes - OOP

In [11]:
class windows:
       
    def dense_rank(self, df):
        df_dense_rank = df.withColumn("dense_rank_flag", dense_rank().over(Window.partitionBy("year").orderBy(desc("total_amount"))))
        return df_dense_rank
    
    def rank(self, df):
        df_rank = df.withColumn("rank_flag", rank().over(Window.partitionBy("year").orderBy(desc("total_amount"))))
        return df_rank
    
    def row_number(self, df):
        df_row_number = df.withColumn("row_flag", row_number().over(Window.partitionBy("year").orderBy(desc("total_amount"))))
        return df_row_number        

In [12]:
obj = windows()

In [13]:
df_result = obj.dense_rank(df)
df_result.show()

+--------+-----------+----------+-------------------+--------+------------+----+-----+---+---------------+
|order_id|customer_id|product_id|         order_date|quantity|total_amount|year|month|day|dense_rank_flag|
+--------+-----------+----------+-------------------+--------+------------+----+-----+---+---------------+
|  O00957|     C01449|     P0498|2023-10-05 00:00:00|       5|      9952.9|2023|   10|  5|              1|
|  O01765|     C01515|     P0498|2023-05-11 00:00:00|       5|      9952.9|2023|    5| 11|              1|
|  O03502|     C00805|     P0498|2023-09-24 00:00:00|       5|      9952.9|2023|    9| 24|              1|
|  O03660|     C01001|     P0498|2023-10-08 00:00:00|       5|      9952.9|2023|   10|  8|              1|
|  O06790|     C01819|     P0498|2023-02-27 00:00:00|       5|      9952.9|2023|    2| 27|              1|
|  O03989|     C00631|     P0440|2023-11-20 00:00:00|       5|     9916.25|2023|   11| 20|              2|
|  O07763|     C00471|     P0440|2023

# Data Writing

In [ ]:
df.write.format("delta")\
    .mode("overwrite")\
    .save("s3://learn-databricks-project-e2e-1-silver/orders")

In [15]:
%sql

CREATE TABLE IF NOT EXISTS learn_e2e_1.silver.orders
USING DELTA
LOCATION 's3://learn-databricks-project-e2e-1-silver/orders'

""
